# Connecting with Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Change working directory
%cd '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry'

/content/drive/MyDrive/Colab Notebooks/deep_acelerometry


# Setup

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, ReLU, Add, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.regularizers import l2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
# Check that we are using GPU
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# Dataset

### 1. Load target labels

In [5]:
target = pd.read_csv('target.csv')

In [6]:
target

,SEQN,DXXNK_TSCORE,CLASSES,TARGET_BINARY
0,73557.0,NaN,NaN,NaN
1,73558.0,-0.358333,normal,0.0
2,73559.0,NaN,NaN,NaN
3,73561.0,-1.133333,osteopenia,1.0
4,73562.0,0.316667,normal,0.0
...,...,...,...,...
3703,83721.0,-0.266667,normal,0.0
3704,83723.0,1.191667,normal,0.0
3705,83724.0,NaN,NaN,NaN
3706,83726.0,0.841667,normal,0.0


### 2. Load relevant features for classification: gender, age and body mass.

In [7]:
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

In [8]:
X_train.shape, X_test.shape

((2003, 9), (501, 9))

In [9]:
X_train.iloc[:,:4]

,SEQN,RIAGENDR,RIDAGEYR,BMXWT
0,76988.0,1.0,67.0,68.3
1,82083.0,2.0,57.0,67.4
2,75435.0,2.0,63.0,60.3
3,76741.0,2.0,68.0,70.1
4,78834.0,2.0,56.0,57.1
...,...,...,...,...
1998,74345.0,1.0,56.0,79.2
1999,82126.0,1.0,56.0,70.5
2000,78475.0,2.0,77.0,71.7
2001,74862.0,2.0,48.0,66.1


In [10]:
# Combine them into a single DataFrame
features_df = pd.concat([X_train.iloc[:,:4], X_test.iloc[:,:4]]).reset_index(drop=True)

In [11]:
features_df

,SEQN,RIAGENDR,RIDAGEYR,BMXWT
0,76988.0,1.0,67.0,68.3
1,82083.0,2.0,57.0,67.4
2,75435.0,2.0,63.0,60.3
3,76741.0,2.0,68.0,70.1
4,78834.0,2.0,56.0,57.1
...,...,...,...,...
2499,79610.0,2.0,66.0,74.3
2500,77717.0,1.0,48.0,131.7
2501,75946.0,2.0,71.0,92.4
2502,75081.0,2.0,66.0,64.7


### 3. Physical activity data (tensors with GAF images)

Choose sampling frequency, which determines image size

In [12]:
#directory = '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry/tensors_288' # 1 sample every 5 minutes
directory = '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry/tensors_144' # 1 sample every 10 minutes
image_size = int(directory.split('_')[2])

# Filenames are read and split them into training and validation sets accordingly.
all_files = os.listdir(directory)
train_files = [f for f in all_files if 'train' in f]
test_files = [f for f in all_files if 'test' in f]


In [13]:
image_size

144

In [14]:
len(train_files)

2003

In [15]:
len(test_files)

501

# Data generator

### Step 1: Define the Generator Class
You'll subclass tf.keras.utils.Sequence to ensure your generator handles batch processing and shuffling properly.

In [16]:
class DataGenerator(Sequence):
    def __init__(self, directory, file_list, target_df, features_df, batch_size=32, shuffle=True):
        self.directory = directory
        self.file_list = file_list
        self.target_df = target_df
        self.features_df = features_df  # Add this to handle the external features
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.file_list))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.floor(len(self.file_list) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_files = [self.file_list[k] for k in indexes]
        return self.__load_data(batch_files)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __load_data(self, batch_files):
        'Loads data for a batch'
        x = []
        y = []
        features = []
        for file_name in batch_files:
            file_path = os.path.join(self.directory, file_name)
            data = np.load(file_path)
            x.append(data)
            subject_id = float(file_name.split('_')[1])
            # Fetch the continuous target value for regression
            target_value = self.target_df[self.target_df['SEQN'] == subject_id]['DXXNK_TSCORE'].iloc[0]
            y.append(target_value)
            # Retrieve additional features
            feature_row = self.features_df[self.features_df['SEQN'] == subject_id][['RIAGENDR', 'RIDAGEYR', 'BMXWT']]
            if not feature_row.empty:
                features.append(feature_row.iloc[0].values)
            else:
                # Handle cases where no features are found
                features.append(np.array([np.nan, np.nan, np.nan]))  # Assuming NaNs as placeholders

        return ([np.array(x), np.array(features)], np.array(y))



### Step 2: Instantiate the Data Generator
Now you can create instances of DataGenerator for both training and validation.

In [17]:
batch_size = 32  # You can adjust this based on your GPU capacity and training needs

# Create instances of DataGenerator for training and testing
train_generator = DataGenerator(directory=directory, file_list=train_files, target_df=target, features_df=features_df, batch_size=batch_size, shuffle=True)
test_generator = DataGenerator(directory=directory, file_list=test_files, target_df=target, features_df=features_df, batch_size=batch_size, shuffle=False)


# Auxiliary function

In [18]:
from tensorflow.keras import backend as K

# Define the R^2 metric
def r2_keras(y_true, y_pred):
    SS_res = K.sum(K.square(y_true - y_pred))
    SS_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return 1 - SS_res / (SS_tot + K.epsilon())

# Compile and train

**Modelo 1)** 5 veces Conv2D + BatchNormalization + MaxPooling

In [22]:
# Define the input layer for the images
image_input = Input(shape=(image_size, image_size, 7))
# Define the input layer for the additional features
features_input = Input(shape=(3,))

# CNN layers - deeper and more complex
nof_filters = 64
x = Conv2D(nof_filters, (3, 3), activation='relu', padding='same')(image_input)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 2, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 4, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 8, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 16, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Dropout(0.1)(x)
x = Flatten()(x)

# Concatenate CNN output and additional features
x = Concatenate()([x, features_input])

# Dense layers
nof_neurons = 64
x = Dense(nof_neurons, activation='relu')(x)
x = Dropout(0.3)(x)
x = BatchNormalization()(x)
output = Dense(1, activation='linear')(x)  # Output layer for regression

# Create and compile the model
model = Model(inputs=[image_input, features_input], outputs=output)
#model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error']) # Metrics for regression
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_squared_error',  r2_keras])


# Add callbacks for early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(train_generator, validation_data=test_generator, epochs=20, callbacks=[early_stopping])


Epoch 1/20
62/62 [==============================] - 26s 361ms/step - loss: 2.1759 - mean_squared_error: 2.1759 - r2_keras: -0.6573 - val_loss: 1.8028 - val_mean_squared_error: 1.8028 - val_r2_keras: -0.1887
Epoch 2/20
62/62 [==============================] - 21s 331ms/step - loss: 1.7637 - mean_squared_error: 1.7637 - r2_keras: -0.3233 - val_loss: 2.5337 - val_mean_squared_error: 2.5337 - val_r2_keras: -0.7374
Epoch 3/20
62/62 [==============================] - 21s 342ms/step - loss: 1.6621 - mean_squared_error: 1.6621 - r2_keras: -0.2143 - val_loss: 1.6455 - val_mean_squared_error: 1.6455 - val_r2_keras: -0.0509
Epoch 4/20
62/62 [==============================] - 21s 331ms/step - loss: 1.5477 - mean_squared_error: 1.5477 - r2_keras: -0.1499 - val_loss: 1.7699 - val_mean_squared_error: 1.7699 - val_r2_keras: -0.1767
Epoch 5/20
62/62 [==============================] - 22s 359ms/step - loss: 1.4512 - mean_squared_error: 1.4512 - r2_keras: -0.0534 - val_loss: 1.6540 - val_mean_squared_err

**Modelo 2)** ResNet

In [20]:
def residual_block(x, filters, kernel_size=3, stride=1, downsample=False):
    shortcut = x
    if downsample:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same')(x)
        shortcut = BatchNormalization()(shortcut)

    x = Conv2D(filters, kernel_size, padding='same', strides=stride)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

def tfm_resnet(input_shape, additional_features_shape):
    image_input = Input(shape=input_shape)
    features_input = Input(shape=additional_features_shape)

    # Initial convolution
    x = Conv2D(64, (7, 7), strides=2, padding='same')(image_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    # ResNet blocks
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2, downsample=True)
    x = residual_block(x, 128)
    x = residual_block(x, 256, stride=2, downsample=True)
    x = residual_block(x, 256)

    # Global Pooling and output layer for image features
    x = GlobalAveragePooling2D()(x)

    # Concatenate CNN output and additional features
    x = Concatenate()([x, features_input])

    # Dense layers and final regression layer
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)
    output = Dense(1, activation='linear')(x)  # Change for regression

    # Create model
    model = Model(inputs=[image_input, features_input], outputs=output)
    return model

# Set up and compile the model for regression
model = tfm_resnet(input_shape=(image_size, image_size, 7), additional_features_shape=(3,))
#model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_squared_error',  r2_keras])

# Train the model with the data generators
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(train_generator, validation_data=test_generator, epochs=20, callbacks=[early_stopping])


Epoch 1/20
62/62 [==============================] - 33s 324ms/step - loss: 2.3128 - mean_squared_error: 2.3128 - r2_keras: -0.7650 - val_loss: 695.9621 - val_mean_squared_error: 695.9621 - val_r2_keras: -492.6993
Epoch 2/20
62/62 [==============================] - 21s 343ms/step - loss: 1.7119 - mean_squared_error: 1.7119 - r2_keras: -0.2787 - val_loss: 154.4236 - val_mean_squared_error: 154.4236 - val_r2_keras: -109.9940
Epoch 3/20
62/62 [==============================] - 20s 322ms/step - loss: 1.5370 - mean_squared_error: 1.5370 - r2_keras: -0.1704 - val_loss: 67.4075 - val_mean_squared_error: 67.4075 - val_r2_keras: -46.6150
Epoch 4/20
62/62 [==============================] - 21s 331ms/step - loss: 1.3733 - mean_squared_error: 1.3733 - r2_keras: -0.0403 - val_loss: 24.7918 - val_mean_squared_error: 24.7918 - val_r2_keras: -16.7602
Epoch 5/20
62/62 [==============================] - 21s 340ms/step - loss: 1.2681 - mean_squared_error: 1.2681 - r2_keras: 0.0481 - val_loss: 1.6836 - val

**Modelo 3)** VGG

In [21]:
# Define the input layer for the images
image_input = Input(shape=(image_size, image_size, 7), name='image_input')
# Define the input layer for the additional features
features_input = Input(shape=(3,), name='features_input')

# Block 1
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1')(image_input)
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

# Block 2
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

# Block 3
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

# Block 4
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)

# Flatten and concatenate additional features
x = Flatten(name='flatten')(x)
x = Concatenate()([x, features_input])

# Dense layers for regression
x = Dense(4096, activation='relu', name='fc1')(x)
x = Dropout(0.5)(x)  # Incorporating dropout for regularization
x = Dense(4096, activation='relu', name='fc2')(x)
x = Dropout(0.5)(x)  # Incorporating dropout for regularization
output = Dense(1, activation='linear', name='predictions')(x)  # Output layer for regression

# Create the model
model = Model(inputs=[image_input, features_input], outputs=output)
#model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_squared_error',  r2_keras])

# Add callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Model summary to check architecture
model.summary()

# Training the model with early stopping
history = model.fit(train_generator, validation_data=test_generator, epochs=20, callbacks=[early_stopping])


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image_input (InputLayer)    [(None, 144, 144, 7)]        0         []                            
                                                                                                  
 block1_conv1 (Conv2D)       (None, 144, 144, 64)         4096      ['image_input[0][0]']         
                                                                                                  
 block1_conv2 (Conv2D)       (None, 144, 144, 64)         36928     ['block1_conv1[0][0]']        
                                                                                                  
 block1_pool (MaxPooling2D)  (None, 72, 72, 64)           0         ['block1_conv2[0][0]']        
                                                                                            